# Modelo de Regresión Lineal — Predicción de Costos Médicos

Este notebook construye y evalúa modelos de regresión lineal para predecir los costos de seguros médicos (`charges`) a partir de las variables `age`, `smoker` y `bmi`.

## Importación de librerías

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Carga y preparación de datos

Se carga el dataset `insurance.csv` y se codifica la variable `smoker` como binaria (1 = fumador, 0 = no fumador).

In [17]:
df_path = r"C:\Users\ivanf\Documents\Vortex\1.3 Vortex - Medical Cost\data\insurance.csv" ##Ubicar r antes de las comillas para evitar errroes de lectura con los backslashs
df_insurance = pd.read_csv(df_path)
df_insurance['smoker'] = (df_insurance['smoker'] == "yes").astype(int)
print(df_insurance)

      age     sex     bmi  children  smoker     region      charges
0      19  female  27.900         0       1  southwest  16884.92400
1      18    male  33.770         1       0  southeast   1725.55230
2      28    male  33.000         3       0  southeast   4449.46200
3      33    male  22.705         0       0  northwest  21984.47061
4      32    male  28.880         0       0  northwest   3866.85520
...   ...     ...     ...       ...     ...        ...          ...
1333   50    male  30.970         3       0  northwest  10600.54830
1334   18  female  31.920         0       0  northeast   2205.98080
1335   18  female  36.850         0       0  southeast   1629.83350
1336   21  female  25.800         0       0  southwest   2007.94500
1337   61  female  29.070         0       1  northwest  29141.36030

[1338 rows x 7 columns]


## 2. División de variables y conjuntos de entrenamiento/prueba

Se definen las variables predictoras (`age`, `smoker`, `bmi`) y la variable objetivo (`charges`). Luego se divide el dataset en 80% entrenamiento y 20% prueba.

In [ ]:
# Definicion de las variables a ingresar en el modelo
x = df_insurance[['age','smoker','bmi']]
# Definicion de las variables objetivo
y = df_insurance['charges']

# Obtension de los data set de prueba y entrenamiento, junto con asignacion del tamaño de test e inclusion de la semilla
x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size=0.2, random_state=42
)

print(f"Training Data Set: {x_train.shape[0]} filas")
print(f"Test Data Set: {x_test.shape[0]} filas")

Training Data Set: 1070 filas
Test Data Set: 268 filas


## 3. Entrenamiento del modelo de regresión lineal

Se instancia y entrena un modelo de regresión lineal con los datos de entrenamiento, y se generan predicciones sobre el conjunto de prueba.

In [20]:
# Creacion del modelo
model = LinearRegression()

# Entrenamiento del modelo, solo con los grupos seleccionados
model.fit(x_train, y_train)

# Prediccion sobre los datos de prueba
y_pred = model.predict(x_test)

## 4. Evaluación del modelo (escala original)

Se calculan las métricas de error sobre las predicciones en escala original:
- **MAE**: error promedio absoluto en dólares.
- **RMSE**: raíz del error cuadrático medio, penaliza errores grandes.
- **R²**: proporción de varianza explicada por el modelo.

In [24]:
# MAE — error promedio absoluto
mae = mean_absolute_error(y_test, y_pred)

# RMSE — error cuadrático medio, castiga errores grandes
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R² — qué porcentaje de la variación de charges explica el modelo
r2 = r2_score(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

MAE:  $4,260.56
RMSE: $5,874.76
R²:   0.7777


## 5. Modelo con transformación logarítmica

Se repite el proceso aplicando `log(charges)` como variable objetivo. Esto estabiliza la varianza y puede mejorar el ajuste cuando los datos tienen distribución sesgada. Las métricas finales se calculan revirtiendo el logaritmo para comparar en dólares reales.

In [28]:
# Definicion de las variables a ingresar en el modelo
x = df_insurance[['age','smoker','bmi']]
# Definicion de las variables objetivo
y = df_insurance['charges']
y_log = np.log(y)

# Obtension de los data set de prueba y entrenamiento, junto con asignacion del tamaño de test e inclusion de la semilla
x_train, x_test, y_train_log, y_test_log = train_test_split(
    x,y_log, test_size=0.2, random_state=42
)

# Creacion del modelo
model_log = LinearRegression()
# Entrenamiento del modelo, solo con los grupos seleccionados
model_log.fit(x_train, y_train_log)
# Prediccion sobre los datos de prueba
y_pred_log = model_log.predict(x_test)

# Revertimos el logaritmo para volver a dólares reales
y_pred_real = np.exp(y_pred_log)
y_test_real = np.exp(y_test_log)

# MAE — error promedio absoluto
mae_log = mean_absolute_error(y_test_real, y_pred_real)
# RMSE — error cuadrático medio, castiga errores grandes
rmse_log = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
# R² — qué porcentaje de la variación de charges explica el modelo
r2_log = r2_score(y_test_real, y_pred_real)

print(f"MAE:  ${mae_log:,.2f}")
print(f"RMSE: ${rmse_log:,.2f}")
print(f"R²:   {r2_log}")

MAE:  $4,215.21
RMSE: $8,556.75
R²:   0.5283826825940188
